# California Housing: regresion de principio a fin

**Pregunta:** usando caracteristicas censales de una zona, ¿cual es su valor mediano de vivienda? La variable objetivo `MedHouseVal` esta expresada en cientos de miles de USD y procede del censo de 1990; su tope historico es 5.0. No representa una tasacion actual.

Seguimos el orden del encargo: datos/pregunta, limpieza y recuento, analisis visual (incluido el mapa), baseline de la media, regresion lineal y Random Forest, comparacion por MAE y guardado del mejor modelo. La particion aleatoria reproducible es apropiada para este problema de regresion no temporal.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from ml_al_alfa.data import clean_california_data
from ml_al_alfa.train import train_and_save

sns.set_theme(style="whitegrid")
Path("artifacts").mkdir(exist_ok=True)

## 1. Carga y pregunta de prediccion

El conjunto se obtiene desde `scikit-learn`. Las coordenadas permiten visualizar la distribucion geografica de la variable objetivo.

In [ ]:
dataset = fetch_california_housing(as_frame=True)
raw = dataset.frame.copy()
print(f"Filas brutas: {len(raw):,}")
print(f"Duplicados: {raw.duplicated().sum():,}")
display(raw.head())
display(raw.describe().T)

## 2. Limpieza

Se eliminan duplicados, nulos, valores no numericos y observaciones fuera de los rangos fisicamente plausibles para el conjunto. El numero de filas antes y despues queda registrado para poder auditar el paso.

In [ ]:
X, y, cleaning_report = clean_california_data(raw)
print(cleaning_report.as_dict())
print("Nulos restantes:", int(X.isna().sum().sum() + y.isna().sum()))
display(X.head())

## 3. Exploracion visual

Los graficos ayudan a detectar asimetria, relaciones no lineales y patrones espaciales antes de entrenar. Las coordenadas se conservan como variables de entrada porque son conocidas en el momento de estimar el valor de la zona.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(y, bins=40, ax=axes[0])
axes[0].set(title="Distribucion del valor mediano", xlabel="MedHouseVal (cientos de miles USD)")
sns.scatterplot(data=X.assign(MedHouseVal=y), x="MedInc", y="MedHouseVal", alpha=0.2, s=12, ax=axes[1])
axes[1].set(title="Ingreso mediano frente a valor de vivienda")
plt.tight_layout()

In [ ]:
fig = px.scatter_geo(
    X.assign(MedHouseVal=y), lat="Latitude", lon="Longitude", color="MedHouseVal",
    color_continuous_scale="Viridis", opacity=0.55,
    title="Mapa de valores medianos de vivienda por zona censal en California",
    hover_data={"MedInc": ":.2f", "MedHouseVal": ":.2f"},
)
fig.update_geos(fitbounds="locations", visible=True, showcountries=False, showsubunits=True)
fig.show()

El mapa y la distribucion dejan claro que la localizacion y el ingreso contienen senales relevantes. El tope de la variable objetivo produce una acumulacion en el limite superior: debe mencionarse al interpretar los errores.

## 4. Baseline, modelos y evaluacion

Se usa una particion aleatoria reproducible 80/20. El baseline predice siempre la media del conjunto de entrenamiento. Los modelos entrenados son regresion lineal con escalado y Random Forest con preprocesamiento integrado. El MAE se expresa en cientos de miles de USD.

In [ ]:
bundle = train_and_save(
    X, y, dataset_name="california", rows=cleaning_report.as_dict()
)
scores = pd.DataFrame(bundle["metrics"]).T.sort_values("mae")
display(scores.style.format({"mae": "{:.4f}", "r2": "{:.4f}"}))
print("Ganador por MAE:", bundle["model_name"])
print("Artefacto guardado en artifacts/california_model.joblib")
if scores.loc[bundle["model_name"], "mae"] < scores.loc["BaselineMedia", "mae"]:
    print("El mejor modelo mejora el baseline en el test.")
else:
    print("El mejor modelo no mejora el baseline; no demuestra valor frente a la media.")

## 5. Conclusion

El MAE es el error medio absoluto por zona censal en cientos de miles de dolares. Para una lectura aproximada en USD, se multiplica por 100.000. Compara siempre el modelo elegido con el baseline; el rendimiento sobre datos censales de 1990 no debe presentarse como una estimacion de mercado actual.